# Prototyp Fazy 3: kontekstowa analiza LLM (spec SPEC.md §6.1 warstwa 3, §7.1, otwarte pytanie #4)

Cel: sprawdzić **realnymi danymi**, czy model językowy wykrywa ataki, które
nasz obecny pipeline (`rules.py` + heurystyka/RandomForest z `ml/train.py`)
**przegapia** -- konkretnie payloady z `EVASIVE_PAYLOADS` w
[ml/dataset.py](../ml/dataset.py), celowo skonstruowane tak, żeby ominąć regex.

**Dwa warianty do wyboru jedną zmienną (`MODEL_SIZE`) w komórce z modelem:**
- `"small"` -- Phi-3-mini-4k-instruct (3.8B, 4-bit) na darmowym T4, zero jednostek płatnych
- `"big"` -- Qwen2.5-32B-Instruct (32B, 4-bit) na A100 (`Runtime -> Change runtime type -> A100 GPU`), płatne jednostki Colaba

Ta sama metodyka, ten sam zestaw testowy, ten sam kod liczący metryki dla obu --
więc wynik małego i dużego modelu jest wprost porównywalny (to samo `df`
na końcu, tylko z innym `MODEL_SIZE`).

To jest test, nie wdrożenie -- wynik ma pomóc odpowiedzieć na otwarte pytanie
#4 specyfikacji (hosting lokalny czy zewnętrzny, jaki rozmiar modelu) *danymi*,
zamiast zgadywaniem.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK -- wlacz T4 w Runtime")
!pip install --quiet transformers accelerate bitsandbytes

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token (scope: repo): ")
REPO = "pamsmediatech-lang/ai-waf-spec"
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/ai-waf-spec 2>&1 | tail -5
%cd /content/ai-waf-spec
import sys
sys.path.insert(0, "/content/ai-waf-spec")

In [ ]:
# Oba modele ungated (nie wymagaja akceptowania licencji na HuggingFace).
# Ustaw MODEL_SIZE na "small" (T4, darmowy) albo "big" (A100, platne
# jednostki) -- reszta notebooka jest identyczna dla obu.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_SIZE = "big"  # "small" | "big"

MODEL_CONFIGS = {
    "small": "microsoft/Phi-3-mini-4k-instruct",   # 3.8B -- T4 16GB, wystarcza z zapasem
    "big": "Qwen/Qwen2.5-32B-Instruct",             # 32B -- wymaga A100 (4-bit: ~20GB VRAM)
}
MODEL_NAME = MODEL_CONFIGS[MODEL_SIZE]

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
print(f"model zaladowany: {MODEL_NAME} (MODEL_SIZE={MODEL_SIZE!r})")

In [ ]:
PROMPT_TEMPLATE = """Jestes analitykiem bezpieczenstwa WAF. Ocen ponizsza wartosc
parametru HTTP (query/body/path) pod katem tego, czy jest to proba ataku
(SQL injection, XSS, path traversal, command injection) czy normalna,
legalna wartosc.

Wartosc: {payload!r}

Odpowiedz DOKLADNIE w formacie:
WERDYKT: ATAK albo BEZPIECZNE
UZASADNIENIE: <jedno zdanie>
"""


def classify_with_llm(payload: str) -> tuple[str, str]:
    prompt = PROMPT_TEMPLATE.format(payload=payload)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    output = model.generate(inputs, max_new_tokens=80, do_sample=False, temperature=None, top_p=None)
    text = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
    verdict = "ATAK" if "WERDYKT: ATAK" in text.upper().replace(":ATAK", ": ATAK") else "BEZPIECZNE"
    return verdict, text.strip()

In [ ]:
# Zestaw testowy: dokladnie te przypadki, ktore w tests/test_ml_dataset.py
# sluza do sprawdzenia, ze zbior NIE jest trywialny -- payloady, ktore
# rules.py (regex) i heurystyka/RandomForest maja szanse przegapic
# (rule_hit_count == 0), plus kilka jednoznacznie benignych zdan do
# sprawdzenia false-positive rate LLM-a.
from app.waf.request import WafRequest
from app.waf.rules import evaluate
from ml.dataset import BENIGN_BODIES, EVASIVE_PAYLOADS, MALICIOUS_PAYLOADS

test_cases = []
for category, payload in EVASIVE_PAYLOADS:
    test_cases.append((payload, 1, category, "evasive"))
for category, payload in MALICIOUS_PAYLOADS[:5]:
    test_cases.append((payload, 1, category, "obvious"))
for benign in BENIGN_BODIES:
    if benign:
        test_cases.append((benign, 0, "benign", "benign"))

print(f"{len(test_cases)} przypadkow testowych")

In [ ]:
results = []
for payload, true_label, category, kind in test_cases:
    req = WafRequest(method="GET", path="/", client_ip="0.0.0.0", query={"x": [payload]})
    rule_hit = len(evaluate(req)) > 0
    verdict, reasoning = classify_with_llm(payload)
    llm_says_attack = verdict == "ATAK"
    results.append({
        "model_size": MODEL_SIZE, "model_name": MODEL_NAME,
        "payload": payload[:50], "kind": kind, "category": category,
        "true_label": "ATAK" if true_label else "BEZPIECZNE",
        "rules_caught_it": rule_hit, "llm_verdict": verdict,
        "llm_correct": llm_says_attack == bool(true_label),
    })
    print(f"[{kind:8s}] rules={rule_hit!s:5s} llm={verdict:10s} prawda={results[-1]['true_label']:10s} :: {payload[:60]}")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"=== Model: {MODEL_NAME} (MODEL_SIZE={MODEL_SIZE!r}) ===\n")
print("=== Ogolna skutecznosc LLM ===")
print(df.groupby("kind")["llm_correct"].mean())

print("\n=== To jest sedno testu: ewazje, ktore regex przegapil (rules_caught_it=False), ")
print("    czy LLM je mimo to zlapal? ===")
evasive_missed_by_rules = df[(df["kind"] == "evasive") & (~df["rules_caught_it"])]
print(evasive_missed_by_rules[["payload", "llm_verdict", "llm_correct"]])
print(f"\nLLM recall na ewazjach ominietych przez regex: "
      f"{evasive_missed_by_rules['llm_correct'].mean():.1%}")

print("\n=== False positive rate LLM na benignych zdaniach ===")
benign_rows = df[df["kind"] == "benign"]
print(f"FP rate: {(1 - benign_rows['llm_correct']).mean():.1%}")

# Zapisz wynik do CSV z sufiksem MODEL_SIZE -- uruchom notebook dwa razy
# (raz "small", raz "big") i porownaj oba pliki, zeby miec twarda
# odpowiedz na otwarte pytanie #4, a nie wrazenie z jednego przebiegu.
df.to_csv(f"/content/ai-waf-spec/notebooks/llm_results_{MODEL_SIZE}.csv", index=False)
print(f"\nZapisano: notebooks/llm_results_{MODEL_SIZE}.csv")

## Jak czytac wynik

- **Wysoki recall na ewazjach + niski FP rate na benignych** -> LLM realnie
  domyka luke z §7.5 (odpornosc na obejscia), warto wdrazac Faze 3 wg planu
  z SPEC.md §10.2 (shadow mode przed wplywem na ruch).
- **Wysoki FP rate** -> model za bardzo "wpada w panike" na zdania
  zawierajace slowa kluczowe SQL w naturalnym jezyku -- potrzeba lepszego
  promptu albo (jesli to `"small"`) wiekszego modelu.
- **Niski recall na ewazjach** -> ten model nie daje realnej przewagi nad
  obecnym pipeline'em na tych konkretnych obejsciach.

Uruchom notebook raz z `MODEL_SIZE = "small"` (T4, darmowe) i raz z
`"big"` (A100, platne), porownaj `notebooks/llm_results_small.csv` vs
`notebooks/llm_results_big.csv` -- jesli duzy model nie daje wyraznie
lepszego recallu na ewazjach niz maly, to argument za "small" jako
docelowym wyborem hostingu (tansze, szybsze, prawie tak samo skuteczne),
nie za "big" tylko dlatego, ze jest wiekszy.